# Does players' recent form predict the map? A rough check

The agent model loses to a one-sentence rule. The plan's second model adds **how the ten
players have been performing lately**. Before spending a day on it, this checks whether there's
any signal at all.

It's deliberately crude: flat averages over each player's whole history. **The cut-off is not
crude:** a map's features only use matches played *before* it, so a match never feeds its own
prediction.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from dataset import TEAM_ALIASES, build_map_table, load_primary, season_folds, split_season
from features import build_features

maps = build_map_table()
learn, test = split_season(maps)

X_agents, y = build_features(learn)
agent_columns = [c for c in X_agents.columns if c.startswith("agent_")]

print(f"learning half: {len(learn):,} maps    (the test half is not touched in this notebook)")

learning half: 756 maps    (the test half is not touched in this notebook)


## 1. Each player's performance, map by map

From `overview.csv`, after the usual two filters (`Map != "All Maps"`, `Side == "both"`):

| Column | What it is |
|---|---|
| `Rating` | overall performance; 1.00 is average |
| `Average Combat Score` | points per round from kills, assists and damage |
| `Average Damage Per Round` | damage per round |
| `Kill, Assist, Trade, Survive %` | share of rounds where the player did something useful (stored as text) |

In [2]:
STATS = ["Rating", "Average Combat Score", "Average Damage Per Round",
         "Kill, Assist, Trade, Survive %"]
SHORT = {"Rating": "rating", "Average Combat Score": "acs",
         "Average Damage Per Round": "adr", "Kill, Assist, Trade, Survive %": "kast"}

overview = load_primary("matches/overview.csv")
per_map = overview["Map"].str.strip().str.lower() != "all maps"
whole_map = overview["Side"] == "both"

players = overview[per_map & whole_map].copy()
players["Team"] = players["Team"].replace(TEAM_ALIASES)
players = players.rename(columns={"Tournament": "tournament", "Stage": "stage",
                                  "Match Type": "match_type", "Match Name": "match_name",
                                  "Map": "map"})

for column in STATS:
    if not pd.api.types.is_numeric_dtype(players[column]):
        players[column] = pd.to_numeric(players[column].astype(str).str.rstrip("%"),
                                        errors="coerce")

match_key = ["tournament", "stage", "match_type", "match_name", "map"]
players = players.merge(
    learn[match_key + ["match_id", "played_at", "team_a", "team_b"]],
    on=match_key, how="inner",
)

print(f"player-map rows in the learning half: {len(players):,}  "
      f"(expected about {len(learn) * 10:,})")
print()
print("how often each stat is missing:")
for stat in STATS:
    print(f"   {stat:35} {players[stat].isna().sum():5} of {len(players):,}")

player-map rows in the learning half: 7,560  (expected about 7,560)

how often each stat is missing:
   Rating                                910 of 7,560
   Average Combat Score                   30 of 7,560
   Average Damage Per Round              910 of 7,560
   Kill, Assist, Trade, Survive %        860 of 7,560


The missing values aren't random: they're almost all Chinese matches.

In [3]:
gaps = players.groupby("tournament").agg(
    rows=("Rating", "size"), rating_missing=("Rating", lambda s: s.isna().sum())
)
gaps["missing %"] = (gaps["rating_missing"] / gaps["rows"] * 100).round(0)

print(gaps[gaps["rating_missing"] > 0].to_string())

                         rows  rating_missing  missing %
tournament                                              
VCT 2025: China Kickoff   560             560      100.0
VCT 2025: China Stage 1  1050             330       31.0
VCT 2025: EMEA Stage 1   1040              20        2.0


Rating, damage and KAST are **missing for all of China Kickoff** and patchy in China Stage 1;
combat score is nearly complete. A rating-only feature knows nothing about part of one region.
Falling back to combat score is the fix, done later in notebook 09.

## 2. Walking forwards through the season

Taking matches in date order, for each one:

1. Work out each side's form **from what's already recorded**: each player's average over
   earlier maps, averaged over the five players.
2. Record the difference, from team A's side.
3. **Only then** add this match to each player's record.

Step 3 comes after step 2, so a match never contributes to its own features. Working per
match also stops map 1 informing map 3 of the same match. A side with no history is left
blank, later filled as "no difference".

In [4]:
totals, counts = {}, {}
form_rows = []

for (_, match_id), match in players.sort_values(["played_at", "match_id"]).groupby(
    ["played_at", "match_id"], sort=True
):
    team_a, team_b = match.iloc[0]["team_a"], match.iloc[0]["team_b"]

    def form_for(team):
        """Average of this side's five players, over their earlier maps only."""
        names = match[match["Team"] == team]["Player"].unique()
        side = {}
        for stat in STATS:
            known = [totals[(p, stat)] / counts[(p, stat)]
                     for p in names if counts.get((p, stat), 0) > 0]
            side[SHORT[stat]] = np.mean(known) if known else np.nan
        side["games"] = np.mean([counts.get((p, "Average Combat Score"), 0) for p in names])
        return side

    side_a, side_b = form_for(team_a), form_for(team_b)

    entry = {"match_id": match_id, "games_diff": side_a["games"] - side_b["games"]}
    for stat in STATS:
        entry[f"{SHORT[stat]}_diff"] = side_a[SHORT[stat]] - side_b[SHORT[stat]]
    form_rows.append(entry)

    # only now does this match count towards anybody's record
    for _, row in match.iterrows():
        for stat in STATS:
            if pd.notna(row[stat]):
                key = (row["Player"], stat)
                totals[key] = totals.get(key, 0.0) + float(row[stat])
                counts[key] = counts.get(key, 0) + 1

form = pd.DataFrame(form_rows).set_index("match_id")
form_columns = [c for c in form.columns if c.endswith("_diff")]

print("how often each feature came out unknown:")
for column in form_columns:
    print(f"   {column:15} {form[column].isna().mean():.0%}")

X_form = learn[["match_id"]].join(form[form_columns], on="match_id").drop(columns="match_id")
X_form = X_form.fillna(0.0)
X_all = pd.concat([X_agents, X_form], axis=1)

print(f"\nform features attached to all {len(X_form):,} maps")

how often each feature came out unknown:
   games_diff      0%
   rating_diff     22%
   acs_diff        11%
   adr_diff        22%
   kast_diff       22%

form features attached to all 756 maps


## 3. Does it help?

The same five folds as notebook 04. Columns are scaled using each fold's training part only,
and the baseline is the full picker rule.

In [5]:
def score(columns):
    """Average accuracy across the five folds, using only these columns."""
    per_fold = []
    for learn_rows, check_rows in season_folds(learn):
        x_train, x_check = X_all.loc[learn_rows, columns], X_all.loc[check_rows, columns]

        # scaled using the training part only, never the part being checked
        middle, spread = x_train.mean(), x_train.std().replace(0, 1)
        x_train, x_check = (x_train - middle) / spread, (x_check - middle) / spread

        model = LogisticRegression(max_iter=5000).fit(x_train, y.loc[learn_rows])
        per_fold.append(model.score(x_check, y.loc[check_rows]))
    return np.array(per_fold)


baseline = []
for _, check_rows in season_folds(learn):
    picked_by_a = learn.loc[check_rows, "map_picked_by"].map({"team_a": 1, "team_b": 0})
    baseline.append((picked_by_a.fillna(1) == y.loc[check_rows]).mean())
baseline = np.array(baseline)

trials = {
    "picker rule (the baseline)": None,
    "agents + map + picker (notebook 04)": list(X_agents.columns),
    "all four form stats": form_columns,
    "rating difference alone": ["rating_diff"],
    "rating + picker": ["rating_diff", "map_picked_by_a"],
    "combat score + picker": ["acs_diff", "map_picked_by_a"],
    "rating + combat score + picker": ["rating_diff", "acs_diff", "map_picked_by_a"],
    "rating + picker + agents": ["rating_diff", "map_picked_by_a"] + agent_columns,
}

print(f"{'':36} {'mean':>6} {'worst':>6} {'best':>6}   per fold")
for name, columns in trials.items():
    s = baseline if columns is None else score(columns)
    beats = "" if columns is None else f"   beats baseline on {int((s > baseline).sum())}/5"
    per_fold = "  ".join(f"{v * 100:4.1f}" for v in s)
    print(f"{name:36} {s.mean() * 100:5.1f}% {s.min() * 100:5.1f}% "
          f"{s.max() * 100:5.1f}%   {per_fold}{beats}")

                                       mean  worst   best   per fold
picker rule (the baseline)            55.5%  50.0%  58.9%   58.8  58.9  50.8  50.0  58.8
agents + map + picker (notebook 04)   51.1%  47.3%  54.3%   51.5  50.0  52.5  54.3  47.3   beats baseline on 2/5


all four form stats                   52.1%  40.3%  60.7%   51.5  40.3  60.7  56.0  51.9   beats baseline on 2/5


rating difference alone               57.5%  53.2%  63.1%   53.7  53.2  63.1  60.3  57.3   beats baseline on 2/5
rating + picker                       57.8%  56.0%  59.6%   59.6  56.5  57.4  56.0  59.5   beats baseline on 4/5
combat score + picker                 57.2%  50.9%  62.6%   60.3  58.9  53.3  50.9  62.6   beats baseline on 4/5
rating + combat score + picker        57.3%  54.3%  60.3%   60.3  56.5  54.9  54.3  60.3   beats baseline on 4/5


rating + picker + agents              53.1%  48.1%  56.9%   51.5  54.8  54.1  56.9  48.1   beats baseline on 2/5


## What this says

**Two columns beat the baseline: who picked the map, and the players' rating difference.**
Together they average **57.8% against 55.5%**, and they stay between 56.0% and 59.6% on every
fold, beating the baseline on four of five. Holding steady across five slices of the season
is the convincing part.

| | |
|---|---|
| Without player form (notebook 04) | 51.4% |
| With player form | **57.8%** |
| Baseline | 55.5% |

**The agent columns still hurt:** adding them drops the pair from 57.8% to 53.1%, the third
separate time.

**Caveats:** this is the best of eight combinations; its +2.3 points is inside the ±4–6 point
margin; rating is unknown for 22% of maps; and flat career averages ignore recent form,
opponents and roster changes. The proper version (notebook 06) handles these.